## Find local max

In [ ]:
#https://stackoverflow.com/questions/48023982/pandas-finding-local-max-and-min
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import signal
from scipy.signal import argrelextrema

def apply_gaussian(sig, filter_size=20):
    # sig: 1D signal
    pad_size = filter_size
    sig_pad = np.pad(sig, pad_size, mode='mean', stat_length=pad_size)
# pad with mean
    win = signal.windows.hann(filter_size)
    return signal.convolve(sig_pad, win,
mode='same')[pad_size:len(sig_pad)-pad_size] / sum(win)

# Generate a noisy AR(1) sample

np.random.seed(0)
rs = np.random.randn(200)
xs = [0]
for r in rs:
    xs.append(xs[-1] * 0.9 + r)
#xs = apply_gaussian(xs)
df = pd.DataFrame(xs, columns=['data'])

n = 10  # number of points to be checked before and after

# Find local peaks

df['min'] = df.iloc[argrelextrema(df.data.values, np.less_equal,
                    order=n)[0]]['data']
df['max'] = df.iloc[argrelextrema(df.data.values, np.greater_equal,
                    order=n)[0]]['data']

# Plot results

plt.scatter(df.index, df['min'], c='r')
plt.scatter(df.index, df['max'], c='g')
plt.plot(df.index, df['data'])
plt.show()

In [ ]:
# Gaussian smoothing with mean padding.
# scipy docs - signal convolve

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

sig = np.repeat([2., 1., 0.], 100)
win = signal.windows.hann(50)
#filtered = signal.convolve(sig, win, mode='same') / sum(win)
filtered = apply_gaussian(sig)

fig, (ax_orig, ax_win, ax_filt) = plt.subplots(3, 1, sharex=True)

ax_orig.plot(sig)
ax_orig.set_title('Original pulse')
ax_orig.margins(0, 0.1)

ax_win.plot(win)
ax_win.set_title('Filter impulse response')
ax_win.margins(0, 0.1)

ax_filt.plot(filtered)
ax_filt.set_title('Filtered signal')
ax_filt.margins(0, 0.1)

fig.tight_layout()
fig.show()

## Load real signal

In [ ]:
episode_i = 0
crop_size = 700

with open(f"data/prediction/tokens_{episode_i:04d}_{crop_size:04d}.npy", "rb") as f:
    decoded_str = str(np.load(f))
    scores = np.load(f) # (12, 1, 1024)

# predict: start_y start_x start_z r0 r1 r2 end_y end_x end_z r0 r1 r2
token_pos = 0
scores = scores[token_pos, 0]
x = np.arange(len(scores))

fig, axs = plt.subplots(1, 1, layout='constrained', figsize=(8,4))
axs.set_title('Token probabilities',fontsize=15)
axs.plot(x, scores, linestyle='-', color='tab:green')
axs.grid(True)
axs.set_xlabel('token',  fontsize=10)
axs.set_ylabel('logits',  fontsize=10)

In [ ]:
# find local extrema
n = 20  # number of points to be checked before and after
argrelextrema(scores, np.greater_equal, order=n)[0]

## Interactive overview

In [ ]:
%matplotlib widget
from ipywidgets import widgets, Layout, interact
from pathlib import Path

crop_size = 1000

# list of files to load
file_list = sorted(Path("data/prediction").glob(f"tokens_*_{crop_size:04d}.npy"))

def update(w: int):
    global fig, ax
    global idx

    episode_i = w
    print("episode", episode_i)
    
    with open(f"data/prediction/tokens_{episode_i:04d}_{crop_size:04d}.npy", "rb") as f:
        decoded_str = str(np.load(f))
        scores_all = np.load(f) # (12, 1, 1024)
    
    # predict: start_y start_x start_z r0 r1 r2 end_y end_x end_z r0 r1 r2
    token_pos = 6
    scores = scores_all[token_pos, 0]
    x = np.arange(len(scores))

    def softmax(x):
        T = 10
        x = np.exp(x / T)
        return x / np.sum(x)

    scores = softmax(scores)

    ax = axs[0]
    ax.clear()
    ax.plot(x, scores, linestyle='-', color='tab:green')
    ax.set_title('end y')
    ax.grid(True)    
    ax.set_xlabel('token',  fontsize=10)
    ax.set_ylabel('logits',  fontsize=10)

    # find local extrema
    n = 30  # number of points to be checked before and after
    max_ind = argrelextrema(scores, np.greater_equal, order=n)[0]
    ax.scatter(max_ind, scores[max_ind], color='red')

    ax = axs[1]
    ax.clear()
    ax.plot(x, scores_all[7, 0], linestyle='-', color='tab:green')
    ax.set_title('end x')
    ax.grid(True)    
    ax.set_xlabel('token',  fontsize=10)
    ax.set_ylabel('logits',  fontsize=10)

    fig.tight_layout()
    

    #max.imshow(image_m)
    fig.canvas.draw_idle()

# dynamic figure
image_width, image_height = 600, 400
dpi = 100
figsize = (image_width / dpi, image_height / dpi)
fig, axs = plt.subplots(2, 1, figsize=figsize, dpi=dpi)
#axs.set_axis_off()

slider_w = widgets.IntSlider(
    min=0, max=len(file_list)-1, step=1, value=0, layout=Layout(width="90%")
)
interact(update, w=slider_w)
pass

## Keyboard interactive plot

In [ ]:
%matplotlib widget
from ipywidgets import widgets, Layout, interact
from pathlib import Path

crop_size = 1000

# list of files to load
file_list = sorted(Path("data/prediction").glob(f"tokens_*_{crop_size:04d}.npy"))

# Only for keyboard control
helpstr = """Keyboard control.
The text field is cleared after you press a letter.
asdf - back 10, back 1, forward 1, forward 10
"""
print(helpstr)
idx = 0

def update(w: int):
    global file_list
    global idx
    global fig, ax

    # Increment episode index
    if w == "":
        return
    else:
        
        if w == "a":
            idx = max(0, idx - 10)
        elif w == "s":
            idx = max(0, idx - 1)
        elif w == "d":
            idx = min(len(file_list)-1, idx+1)
        elif w == "f":
            idx = min(len(file_list)-1, idx+10)

    # Reset text field
    text_w.value = ""

    episode_i = idx
    print("episode", episode_i, "of", len(file_list))
    
    
    with open(f"data/prediction/tokens_{episode_i:04d}_{crop_size:04d}.npy", "rb") as f:
        decoded_str = str(np.load(f))
        scores_all = np.load(f) # (12, 1, 1024)
    
    # predict: start_y start_x start_z r0 r1 r2 end_y end_x end_z r0 r1 r2
    # plot 1
    token_pos = 0
    scores = scores_all[token_pos, 0]
    x = np.arange(len(scores))

    def softmax(x):
        T = 10
        x = np.exp(x / T)
        return x / np.sum(x)

    scores = softmax(scores)

    ax = axs[0]
    ax.clear()
    ax.plot(x, scores, linestyle='-', color='tab:green')
    ax.set_title('start y')
    ax.grid(True)    
    ax.set_xlabel('token',  fontsize=10)
    ax.set_ylabel('logits',  fontsize=10)

    # find local extrema
    n = 30  # number of points to be checked before and after
    max_ind = argrelextrema(scores, np.greater_equal, order=n)[0]
    ax.scatter(max_ind, scores[max_ind], color='red')

    # plot 2
    scores = scores_all[1, 0]
    
    ax = axs[1]
    ax.clear()
    ax.plot(x, scores, linestyle='-', color='tab:green')
    ax.set_title('start x')
    ax.grid(True)    
    ax.set_xlabel('token',  fontsize=10)
    ax.set_ylabel('logits',  fontsize=10)
    fig.tight_layout()

    # find local extrema
    n = 30  # number of points to be checked before and after
    max_ind = argrelextrema(scores, np.greater_equal, order=n)[0]
    ax.scatter(max_ind, scores[max_ind], color='red')
    
    fig.canvas.draw_idle()
    

# dynamic figure
image_width, image_height = 600, 400
dpi = 100
figsize = (image_width / dpi, image_height / dpi)
fig, axs = plt.subplots(2, 1, figsize=figsize, dpi=dpi)
#axs.set_axis_off()

text_w = widgets.Text(
    value="",
    placeholder="Type something",
    description="String:",
    disabled=False   
)

interact(update, w=text_w)
pass